In [1]:
from utility.plot_spectrum import plot_spectrum
from utility.adjust_redshift import adjust_redshift
from utility.my_colors import palette_light, pink

from processing.remove_spikes import remove_spikes
from processing.fit_continuum import fit_continuum
from processing.quality_cuts import quality_cuts
from processing.make_bal_mask import make_bal_mask
from processing.fit_gaussians import fit_single_gaussian, fit_two_gaussians, ftest_which_gaussian
from processing.measure_line import measure_line
from processing.parameters import CIV_CONTINUUM_WINDOWS
from processing.get_absorption_intervals import get_absorption_intervals

from data.local.hamann_erq_desi_idxs import valid_sdss_names

from sparcl.client import SparclClient
from dl import queryClient as qc
from astropy.table import Table
from astropy.convolution import convolve, Gaussian1DKernel
from scipy.interpolate import interp1d
from datetime import datetime
from scipy.signal import argrelmin, argrelmax
from ast import literal_eval

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import warnings
import json
import os

hamann_fits_path = "data/local/ERQs_core_all_MNRAS.fits"
search_log_path = "data/local/hamann-objects-desi-search-log.json"
matched_desi_spectra_path = "data/local/matched_desi_spectra.json"
found_desi_spectra_stats_path = "data/found_desi_spectra_stats.json"

pd.options.mode.chained_assignment = None

### load data

In [2]:
hamann_data = Table.read("data/local/ERQs_core_all_MNRAS.fits").to_pandas() #pd.read_csv(hamann_erq_path,index_col=0)

hamann_data["sdss_name"] = hamann_data["sdss_name"].astype("string")

match_df = pd.read_json(search_log_path)
match_df = match_df[["SDSS_id","best_match_id"]].copy()
match_df["SDSS_id"] = match_df["SDSS_id"].str[1:]

match_df = match_df.rename(columns={"best_match_id":"targetid", "SDSS_id":"sdss_name"})

hamann_erq = hamann_data.merge(match_df,on="sdss_name",how="left")

hamann_erq.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 97 entries, 0 to 96
Data columns (total 47 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   sdss_name    97 non-null     object 
 1   Plate        97 non-null     int32  
 2   MJD          97 non-null     int32  
 3   FiberID      97 non-null     int32  
 4   ThingID      97 non-null     int32  
 5   z_dr12       97 non-null     float64
 6   i            97 non-null     float64
 7   i_err        97 non-null     float64
 8   W3           97 non-null     float64
 9   W3_snr       97 non-null     float64
 10  cc_flags     97 non-null     object 
 11  bal_flag_vi  97 non-null     int32  
 12  f1450        97 non-null     float64
 13  alpha_civ    97 non-null     float64
 14  alpha_nv     97 non-null     float64
 15  alpha_all    97 non-null     float64
 16  alphae_all   97 non-null     float64
 17  alpha_allc   97 non-null     float64
 18  alphae_allc  97 non-null     float64
 19  rew       

In [3]:
# client = SparclClient()

# idxs = [int(x) for x in hamann_erq["targetid"] if x != "Not found"]

# output_fields = ['specid', 'redshift', 'wavelength', 'flux', 'ivar', 'spectype', 'targetid', 'redshift_warning']

# spectra = client.retrieve_by_specid(idxs, dataset_list=["DESI-DR1"], include=output_fields)

In [4]:
# spectra_df = pd.DataFrame(spectra.data[1:])

# spectra_df.to_json(matched_desi_spectra_path,default_handler=str)

# spectra_df

### measure lines

In [ ]:
# spectra_df = pd.read_json(matched_desi_spectra_path)

# stats_half = hamann_erq[['sdss_name', 'rew', 'fwhm', 'kt80', 'targetid']].copy()

# spectra_df["targetid"] = spectra_df["targetid"].astype("string")

# data = spectra_df.merge(stats_half,on="targetid")

# data = data[data["sdss_name"].isin(valid_sdss_names)]

# data = data.drop_duplicates(subset="sdss_name")

# data.to_json(found_desi_spectra_stats_path)

In [16]:
data = pd.read_json(found_desi_spectra_stats_path)

data["my_fwhm"] = None
data["my_rew"] = None
data["fit_rew"] = None
data["my_kt80"] = None

In [8]:
def get_ci(lam,profile,interval=0.99):
    cutoff = (1-interval)/2
    
    cumsum = np.cumsum(profile)
    cumsum /= np.max(cumsum)
    
    lower_idx = np.searchsorted(cumsum,cutoff)
    upper_idx = np.searchsorted(cumsum,1-cutoff,side="left")

    return (lam[lower_idx], lam[upper_idx])

In [9]:
def sanity_tests(idx):
    lam = np.array(data["wavelength"][idx])
    flux = np.array(data["flux"][idx])
    ivar = np.array(data["ivar"][idx])
    z = data["redshift"][idx]

    lam, flux, ivar = adjust_redshift(z, lam, flux, ivar=ivar)

    # flux = convolve(flux, Gaussian1DKernel(1))

    flux_clean, spike_mask = remove_spikes(lam,flux,ivar=ivar,sigma_thresh=3)

    result, continuum, window_mask = fit_continuum(lam, flux_clean, ivar=ivar)

    rejected, reason, flags = quality_cuts(lam, flux_clean, continuum, result, max_slope=10)

    if rejected:
        print(f'SDSS: {data["sdss_name"][idx]}\nRejected because: {reason}')
        plot_spectrum(lam, flux_clean, add_smoothed=False)
        return 

    # flux_clean = convolve(flux_clean, Box1DKernel(5))

    flux_sub = flux_clean - continuum if not rejected else None

    absorption_interval = get_absorption_intervals(lam,flux,ivar,continuum)

    if absorption_interval is not None:
        if len(absorption_interval) == 1:
            bal_mask = (lam > absorption_interval[0][0]) & (lam < absorption_interval[0][1])
        elif len(absorption_interval) == 2:
            bal_mask = ((lam > absorption_interval[0][0]) & (lam < absorption_interval[0][1])) | ((lam > absorption_interval[1][0]) & (lam < absorption_interval[1][1]))
    else:
        bal_mask = None
    # bal_mask = make_bal_mask(lam, flux_sub, ivar=ivar, min_width_aa=5)

    result1, profile1 = fit_single_gaussian(lam, flux_sub, ivar=ivar, mask=bal_mask)
    result2, profile2 = fit_two_gaussians(lam, flux_sub, ivar=ivar, mask=bal_mask, single_result=result1)

    accept_two, p_val = ftest_which_gaussian(result1, result2)

    if accept_two:
        # sigma_w = result2.params["sigma_w"].value
        # sigma_c = result2.params["sigma_c"].value
        # mu_w = result2.params["cen_w"].value
        # mu_c = result2.params["cen_c"].value

        # lower = min(mu_c-3*sigma_c,mu_w-3*sigma_w)
        # upper = max(mu_c+3*sigma_c,mu_w+3*sigma_w)
        # ci = (lower,upper)

        profile = profile2
    else:
        # mu = result1.params["center"].value
        # sigma = result1.params["sigma"].value
        # ci = (mu-3*sigma, mu+3*sigma)
        
        profile = profile1

    # ci = (mu-2*sigma, mu+2*sigma)
    ci = get_ci(lam,profile)

    plot_mask = (lam >= 1400) & (lam <= 1700)
    continuum_mask = plot_mask#(lam >= 1425) & (lam <= 1810)

    # flux_sub = convolve(flux_sub, Box1DKernel(7))

    if absorption_interval is not None:
        line_stats, win = measure_line(lam, profile, flux_sub, continuum, ci, get_window=True)
        plot_spectrum(lam[plot_mask], flux[plot_mask], flux_label="flux",
            line_width_factor=2,
            add_smoothed=False,
            additional_lam=[lam[continuum_mask]]*2,
            additional_flux=[profile2[continuum_mask]+continuum[continuum_mask],continuum[continuum_mask]],
            additional_flux_label=["line fit","continuum fit"],
            line_lambda=[1549],
            line_label=["CIV"],
            shaded_regions=CONTINUUM_WINDOWS[:2]+[(lam[win][0], lam[win][-1]), *absorption_interval],
            shaded_regions_colors=["green"]*2+["orange"]+["DodgerBlue"]*len(absorption_interval),
            shaded_regions_label=["continuum fit"]*2+["integration"]+["absorption mask"]*len(absorption_interval)
            )
    else:
        line_stats, win = measure_line(lam, profile, flux_sub, continuum, ci, get_window=True)
        plot_spectrum(lam[plot_mask], flux[plot_mask], flux_label="flux",
            line_width_factor=2,
            add_smoothed=False,
            additional_lam=[lam[continuum_mask]]*2,
            additional_flux=[profile[continuum_mask]+continuum[continuum_mask],continuum[continuum_mask]],
            additional_flux_label=["line fit","continuum fit"],
            line_lambda=[1549],
            line_label=["CIV"],
            shaded_regions=CONTINUUM_WINDOWS[:2]+[(lam[win][0], lam[win][-1])],
            shaded_regions_colors=["green"]*2+["orange"],
            shaded_regions_label=["continuum fit"]*2+["EW integration"]
            )

    out_str = f'''
                SDSS: {data["sdss_name"][idx]}
                FWHM: {round(line_stats["fwhm_kms"],3)} vs {round(data["fwhm"][idx],3)} (Hamann)
                REW: {round(line_stats["ew_aa"],3)} vs {round(data["rew"][idx],3)} (Hamann) vs {round(line_stats["fit_rew"],3)} (fit area)
                kt80: {round(line_stats["kt80"],3)} vs {round(data["kt80"][idx],3)} (Hamann)
    '''

    print(out_str)

    data["my_fwhm"][idx] = line_stats["fwhm_kms"]
    data["my_rew"][idx] = line_stats["ew_aa"]
    data["fit_rew"][idx] = line_stats["fit_rew"]
    data["my_kt80"][idx] = line_stats["kt80"]
    

In [11]:
# for i in range(len(data)):
#     print("---------------------------------------\n---------------------------------------")
#     sanity_tests(i)

In [12]:
# stats = data[data["my_rew"].notnull()][['sdss_name','redshift', 'rew', 'fit_rew', 'fwhm', 'kt80', 'my_fwhm', 'my_rew', 'my_kt80']]

# stats["rew_abs_diff"] = stats["my_rew"] - stats["rew"]
# stats["fwhm_abs_diff"] = stats["my_fwhm"] - stats["fwhm"]
# stats["kt80_abs_diff"] = stats["my_kt80"] - stats["kt80"]

# stats["rew_rel_diff"] = stats["rew_abs_diff"] / stats["rew"]
# stats["fwhm_rel_diff"] = stats["fwhm_abs_diff"] / stats["fwhm"] 
# stats["kt80_rel_diff"] = stats["kt80_abs_diff"] / stats["kt80"]

# stats

In [13]:
# current_stats_path = "data/local/desi_current_best_measurements.csv"

# stats.to_csv(current_stats_path)